## Install required dependencies

In [ ]:
%pip install 'ultralytics>=8.3.0' roboflow wandb matplotlib

## Inject secrets into the environment

In [22]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("roboflow_api_key")
dataset_version = user_secrets.get_secret("dataset_version")
project_name = user_secrets.get_secret("project_name")
wandb_api_key = user_secrets.get_secret("wandb_api_key")
wandb_username = user_secrets.get_secret("wandb_username")
workspace_name = user_secrets.get_secret("workspace_name")

## Download dataset

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace(workspace_name).project(project_name)
dataset = project.version(dataset_version).download("yolov11")

## Adjust paths in the config file

In [23]:
import yaml

new_paths = {
    "test": "./test/images",
    "train": "./train/images",
    "val": "./valid/images"
}

def modify_paths_in_yaml(file_path):
    with open(file_path, "r") as file:
        data = yaml.safe_load(file)

    data["test"] = new_paths["test"]
    data["train"] = new_paths["train"]
    data["val"] = new_paths["val"]

    with open(file_path, "w") as file:
        yaml.dump(data, file)

file_path = f"./{project_name}-{dataset_version}/data.yaml"

modify_paths_in_yaml(file_path)

## Train the model

In [ ]:
import wandb
from ultralytics import YOLO

yolo_version = "yolo11n"
epochs_num = 50
image_size = 640
batch_size = 16

wandb.login(key=wandb_api_key)

model = YOLO(f"{yolo_version}.pt")

model.train(
    project=project_name,
    name=yolo_version,
    data=f"/kaggle/working/{project_name}-{dataset_version}/data.yaml",
    epochs=epochs_num,
    imgsz=image_size,
    batch=batch_size
)

## Validate the model

In [ ]:
val = model.val(
    data=f"/kaggle/working/{project_name}-{dataset_version}/data.yaml",
    imgsz=image_size,
    batch=batch_size
)

## Deploy the model

In [ ]:
project.version(dataset_version).deploy(
    model_type="yolov11",
    model_path=f"/kaggle/working/{project_name}/{yolo_version}"
)